In [ ]:
!git clone https://github.com/Eben113/DeblurGAN

Cloning into 'DeblurGAN'...
remote: Enumerating objects: 373, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 373 (delta 7), reused 0 (delta 0), pack-reused 356 (from 1)
Receiving objects: 100% (373/373), 85.87 MiB | 28.40 MiB/s, done.
Resolving deltas: 100% (196/196), done.


In [ ]:
%cd DeblurGAN

/content/DeblurGAN


In [ ]:
import sys
from torch import nn
import cv2
from torchvision import transforms, datasets
import numpy as np
from torch.utils.data import Dataset, DataLoader
import cv2
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow
import skimage.io as io
import os
import torch
from options.train_options import TrainOptions
from PIL import Image, ImageFilter
import glob

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ronakgohil/license-plate-dataset")

print("Path to dataset files:", path)

100%|██████████| 147M/147M [00:01<00:00, 107MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ronakgohil/license-plate-dataset/versions/2


In [ ]:
#!ls /root/.cache/kagglehub/datasets/ronakgohil/license-plate-dataset/versions/2/archive/images
data_path = "/root/.cache/kagglehub/datasets/ronakgohil/license-plate-dataset/versions/2/archive/images/Btrain/train"
val_path = "/root/.cache/kagglehub/datasets/ronakgohil/license-plate-dataset/versions/2/archive/images/Bval/val"

In [ ]:
%mkdir "/root/.cache/kagglehub/datasets/ronakgohil/license-plate-dataset/versions/2/archive/images/Atrain/train_blurred"
%mkdir "/root/.cache/kagglehub/datasets/ronakgohil/license-plate-dataset/versions/2/archive/images/Aval/val_blurred"

In [ ]:
train_blurred_folder = "/root/.cache/kagglehub/datasets/ronakgohil/license-plate-dataset/versions/2/archive/images/Atrain/train_blurred"
val_blurred_folder = "/root/.cache/kagglehub/datasets/ronakgohil/license-plate-dataset/versions/2/archive/images/Aval/val_blurred"

In [ ]:
def create_blurred_fold(orig_Fold, blrd_Fold):
  for name in glob.glob(orig_Fold + "/*.jpg"):
    img = cv2.imread(name)
    blrd = cv2.medianBlur(img, 3)
    imgName = os.path.split(name)[-1]
    cv2.imwrite(blrd_Fold + "/" + imgName, blrd)

In [ ]:
create_blurred_fold(data_path, train_blurred_folder)

In [ ]:
create_blurred_fold(val_path, val_blurred_folder)

In [ ]:
%mkdir AB

In [ ]:
!python datasets/combine_A_and_B.py --fold_A /root/.cache/kagglehub/datasets/ronakgohil/license-plate-dataset/versions/2/archive/images/Atrain --fold_B /root/.cache/kagglehub/datasets/ronakgohil/license-plate-dataset/versions/2/archive/images/Btrain --fold_AB /content/DeblurGAN/AB

[fold_A] =  /root/.cache/kagglehub/datasets/ronakgohil/license-plate-dataset/versions/2/archive/images/Atrain
[fold_B] =  /root/.cache/kagglehub/datasets/ronakgohil/license-plate-dataset/versions/2/archive/images/Btrain
[fold_AB] =  /content/DeblurGAN/AB
[num_imgs] =  1000000
[use_AB] =  False
split = train_blurred, use 1526/1526 images
split = train_blurred, number of images = 1526


In [ ]:
plt.imread(glob.glob(data_path+"/*")[5]).shape

(1076, 1644, 3)

In [ ]:
from DeblurGAN.models import conditional_gan_model

In [ ]:
pretrained_gen = torch.load("checkpoints/experiment_name/latest_net_G.pth")
pretrained_disc = torch.load("checkpoints/experiment_name/latest_net_D.pth")

In [ ]:
len(pretrained_gen.keys())

72

In [ ]:
from models import models
options = TrainOptions()
options.parser.add_argument("-f")
parsed = options.parse()
model = models.create_model(parsed)

------------ Options -------------
batchSize: 1
beta1: 0.5
checkpoints_dir: ./checkpoints
continue_train: False
dataroot: D:\Photos\TrainingData\BlurredSharp\combined
dataset_mode: aligned
display_freq: 100
display_id: 1
display_port: 8097
display_single_pane_ncols: 0
display_winsize: 256
epoch_count: 1
f: /root/.local/share/jupyter/runtime/kernel-a3e7c8a9-be51-4ea3-885c-910fb764c0b8.json
fineSize: 256
gan_type: wgan-gp
gpu_ids: [0]
identity: 0.0
input_nc: 3
isTrain: True
lambda_A: 100.0
lambda_B: 10.0
learn_residual: False
loadSizeX: 640
loadSizeY: 360
lr: 0.0001
max_dataset_size: inf
model: content_gan
nThreads: 2
n_layers_D: 3
name: experiment_name
ndf: 64
ngf: 64
niter: 150
niter_decay: 150
no_dropout: False
no_flip: False
no_html: False
norm: instance
output_nc: 3
phase: train
pool_size: 50
print_freq: 100
resize_or_crop: resize_and_crop
save_epoch_freq: 5
save_latest_freq: 5000
serial_batches: False
which_direction: AtoB
which_epoch: latest
which_model_netD: basic
which_model_net

/content/DeblurGAN/models/conditional_gan_model.py:25: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:78.)
  self.input_A = self.Tensor(opt.batchSize, opt.input_nc,  opt.fineSize, opt.fineSize)
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.

---------- Networks initialized -------------
ResnetGenerator(
  (model): Sequential(
    (0): ReflectionPad2d((3, 3, 3, 3))
    (1): Conv2d(3, 64, kernel_size=(7, 7), stride=(1, 1))
    (2): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=True)
    (3): ReLU(inplace=True)
    (4): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (5): InstanceNorm2d(128, eps=1e-05, momentum=0.1, affine=False, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (8): InstanceNorm2d(256, eps=1e-05, momentum=0.1, affine=False, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): ResnetBlock(
      (conv_block): Sequential(
        (0): ReflectionPad2d((1, 1, 1, 1))
        (1): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1))
        (2): InstanceNorm2d(256, eps=1e-05, momentum=0.1, affine=False, track_running_stats=True)
        (3): ReLU(inplace=True)
      

In [ ]:
model.netG.load_state_dict(pretrained_gen)

<All keys matched successfully>

In [ ]:
model.netD.load_state_dict(pretrained_disc)

<All keys matched successfully>

In [ ]:
class License_data(Dataset):
  def __init__(self, state = "train"):
    self.state = state
    if self.state == "train":
      self.path = data_path
    else:
      self.path = val_path
  def __len__(self):
    return len(os.listdir(self.path))
  def __getitem__(self, index):
    imgName = os.listdir(self.path)[index]
    arr = plt.imread(self.path + "/" + imgName)
    arr = cv2.resize(arr, (640, 360))
    if self.state == "train":
      blurred = cv2.medianBlur(arr, 3)
      total = np.concatenate((blurred, arr), 1)
      return torch.from_numpy(total.astype(dtype = "float32"))
    return torch.from_numpy(arr)

In [ ]:
train_test = License_data(state = "train")
val = License_data(state = "val")
trainloader = DataLoader(train_test, shuffle = True, batch_size=30)
valLoader = DataLoader(val, shuffle = True, batch_size = 1)

In [ ]:
class custom_loader():
  def __init__(self, loader):
    self.loader = loader
  def load_data(self):
    return self.loader

In [ ]:
arr = train_test[5][:,0:640]

In [ ]:
arr.shape

torch.Size([360, 640, 3])

In [ ]:
transform = [transforms.Resize((640,360), Image.BICUBIC), transforms.RandomCrop(256), transforms.Normalize((0.5, 0.5, 0.5),
                          (0.5, 0.5, 0.5))]
composed = transforms.Compose(transform)

cropped = transforms.Compose([transforms.Resize((640,360), Image.BICUBIC), transforms.RandomCrop(256)])

In [ ]:
img = Image.open("/content/DeblurGAN/images/yolo_b.jpg").convert('RGB')
img_tensor = transforms.functional.pil_to_tensor(img).float()
processed = composed(img_tensor)